# Calibracao da camera do robo PMR3502

**Atividade fora do notebook.**

1. Entre no ambiente virtual Python usado no robo.
2. Instale a versao de OpenCV indicada na apostila.


```bash
wget https://lsc.poli.usp.br/pmr3502/opencv_contrib_python-4.11.0.86-cp311-cp311-linux_aarch64.whl
pip install opencv_contrib_python-4.11.0.86-cp311-cp311-linux_aarch64.whl
sudo apt install -y libhdf5-103 libvtk9.1 libgoogle-glog0v6 libtesseract5 libgstreamer1.0-0 libgstreamer-plugins-base1.0-0
```

3. Instale as bibliotecas de suporte do sistema.

```bash
sudo apt install -y libhdf5-103 libvtk9.1 libgoogle-glog0v6 libtesseract5 libgstreamer1.0-0 libgstreamer-plugins-base1.0-0
```
4. Edite o arquivo `pyvenv.cfg` do ambiente virtual e troque `include-system-site-packages = false` por `include-system-site-packages = true`.


```text
include-system-site-packages = false
```
para:

```text
include-system-site-packages = true
```

5. Reinicie o ambiente virtual.

6. Verifique que o ambiente tem acesso aos pacotes globais com

```bash
python3 -c ’import sys; print(sys.path)’
```
Você deve ver entre os diretórios listados o caminho /usr/lib/python3/dist-packages

7. Instale globalmente a biblioteca `picamera2`.


```bash
sudo apt install -y python3-picamera2 --no-install-recommends
```

8. Imprima o padrao `checkerboard_radon.png` da OpenCV em uma folha A4 sem margens. O padrao deve ficar centralizado e cada quadrado deve medir **12,1 mm**.




## 6.2 Acessando a camera via PiCamera2

Teste simples de captura em `/tmp/imagem1.jpg`.

In [ ]:
import picamera2

with picamera2.Picamera2() as camera:
    camera.start_and_capture_file('/tmp/imagem1.jpg', show_preview=False)

# Se estiver usando a camera NoIR, use a alternativa abaixo (Acho que não é o nosso caso):
#
# with picamera2.Picamera2(tuning='/usr/share/libcamera/ipa/rpi/vc4/ov5647_noir.json') as camera:
#     camera.start_and_capture_file('/tmp/imagem1.jpg', show_preview=False)

Abra e verifique a imagem

Teste continuo da camera - CapturaPicamera2.ipynb. O código do git já está passado aqui abaixo: \
Deveria mostrar imagens da camera continuamente

In [ ]:
from picamera2 import Picamera2
from matplotlib import pyplot as plt
from IPython.display import clear_output
import os

os.environ["LIBCAMERA_LOG_LEVELS"] = "3"
Picamera2.set_logging(Picamera2.ERROR)

with Picamera2(tuning=os.environ.get('LIBCAMERA_RPI_TUNING_FILE', None)) as camera:
    try:
        cfg = camera.create_video_configuration(main={"size": (1296, 972), "format": "BGR888"})
        camera.configure(cfg)
        camera.start()
        while True:
            img = camera.capture_array()
            clear_output(wait=True)
            plt.imshow(img)
            plt.show()
    except KeyboardInterrupt:
        pass

Imports e constantes usados nas celulas seguintes.

In [ ]:
from pathlib import Path
import glob
import json
import math
import os

import cv2
import numpy as np
from matplotlib import pyplot as plt

IMAGE_WIDTH = 1296
IMAGE_HEIGHT = 972
IMAGE_SIZE = (IMAGE_WIDTH, IMAGE_HEIGHT)

SQUARE_SIZE_MM = 12.1
CAPTURAS_INTRINSECAS_DIR = Path.cwd() / 'capturas_intrinsecos'
CAPTURA_EXTRINSECA_PATH = Path.cwd() / 'captura_extrinseca.jpg'
RESULTS_PATH = Path.cwd() / 'camera_calibration_results.npz'

CHESSBOARD_FLAGS = (
    cv2.CALIB_CB_EXHAUSTIVE
    + cv2.CALIB_CB_LARGER
    + cv2.CALIB_CB_MARKER
)

np.set_printoptions(precision=6, suppress=True)

Capture um novo frame adicionando a seguite celula:

In [ ]:
with Picamera2(tuning=os.environ.get('LIBCAMERA_RPI_TUNING_FILE', None)) as camera:
    cfg = camera.create_video_configuration(main={"size": (1296, 972), "format": "BGR888"})
    camera.configure(cfg)
    camera.start(show_preview=False)
    frame_picam2 = camera.capture_array()

Teste da instalacao de OpenCV. Gravar uma imagem em arquivo usando picamera2 e recupera-la com OpenCV.

In [ ]:
import cv2

filename = '/tmp/img2cv.jpg'
with Picamera2(tuning=os.environ.get('LIBCAMERA_RPI_TUNING_FILE', None)) as camera:
    cfg = camera.create_video_configuration(main={"size": (1296, 972)})
    camera.configure(cfg)
    camera.start_and_capture_file(filename, show_preview=False)
frame_cv2 = cv2.imread(filename)

Mostre a imagem capturada com picamera com o comando:

In [ ]:
# Imagem capturada usando Picamera2
plt.imshow(frame_picam2)
plt.show()

# Imagem recuperada usando OpenCV
plt.imshow(frame_cv2)
plt.show()

print(type(frame_cv2))
print(frame_cv2.dtype)
print(frame_cv2.shape)

print(type(frame_picam2))
print(frame_picam2.dtype)
print(frame_picam2.shape)

plt.imshow(cv2.cvtColor(frame_cv2, cv2.COLOR_BGR2RGB))
plt.show()

## 6.6 Obtendo os parametros Intrinsecos da camera com OpenCV

**Atividade fora do notebook.**

A apostila pede recuperar e executar o programa de captura:

```bash
git clone https://gitlab.uspdigital.usp.br/thiago/pmr3502-captura-de-imagens.git
```
O que já foi feito e salvo na pasta pmr3502-captura-de-imagens-main \
Como o codigo ja esta disponivel nesta pasta para habilitar o serviço use:

```bash
cd pmr3502-camera-calibration/pmr3502-captura-de-imagens-main/pmr3502-captura-de-imagens-main
python3 camera_stream.py -d ../../capturas_intrinsecos -p 8080
```

Este serviço permite, através da url disponibilizada, tirar "fotos" com a camera e salva-las no endereço especificado em -d.

Passo a passo da coleta de imagens:

1. Posicione uma impressao do padrao de calibracao em uma superficie plana de cor clara.

2. Execute o programa de captura e abra a pagina por ele publicada em um browser.

3. Segure o robo com as maos e tente enquadrar com a camera o padrao quadriculado de forma que ele ocupe a maior parte da imagem.

4. Capture imagens do padrao em diversas orientacoes.

5. Procure manter os quatro quadrados centrais, incluindo os 3 com marcas circulares, sempre visiveis.

6. Inclua angulos nos quais o eixo da camera faz mais de 45 graus com a direcao normal do plano do padrao.

7. Obtenha aproximadamente 50 imagens.

8. Interrompa o programa e verifique se as imagens capturadas de fato correspondem a imagens do padrao de calibracao.

Vamos agora agrupar todas as imagens, convertê-las em tons de cinza e utilizar a função cv2.findChessboardCornersSBWithMeta() para encontrar os pontos chave na imagem.

In [ ]:
arquivos_imagens = glob.glob(str(CAPTURAS_INTRINSECAS_DIR / '*.jpg'))

pontos_imagem = []
rotulos = []
imagens_casadas = []

for arquivo in arquivos_imagens:
    img = cv2.imread(arquivo)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners, meta = cv2.findChessboardCornersSBWithMeta(
        gray,
        (6, 6),
        flags=cv2.CALIB_CB_EXHAUSTIVE + cv2.CALIB_CB_LARGER + cv2.CALIB_CB_MARKER,
    )
    if ret: # ret é o booleano que indica se o padrão foi encontrado ou não
        pontos_imagem.append(corners) # coordenadas dos cantos encontrados
        rotulos.append(meta) 
        imagens_casadas.append(img) # imagens onde o padrão foi encontrado

print('Imagens encontradas:', len(arquivos_imagens))
print('Padroes encontrados:', len(pontos_imagem))

Verifique que padroes foram encontrados em ao menos 30 imagens. Aproveitando para visualizar os padrões em todas as imagens.

In [ ]:
print('Padroes encontrados:', len(pontos_imagem))

cols = 3
lines = (len(imagens_casadas) + cols - 1) // cols
fig, axes = plt.subplots(lines, cols, figsize=(2 + 6 * cols, 5 * lines + 2))
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis('off')

for i, c in enumerate(pontos_imagem):
    nova_imagem = np.array(imagens_casadas[i])
    cv2.drawChessboardCorners(nova_imagem, rotulos[i].shape, c, True)
    axes[i//cols, i%cols].imshow(nova_imagem)

fig.show()

Crie as matrizes de coordenadas para cada padrao encontrado e coloque-as em uma sequencia.

In [ ]:
pontos_mundo = []

for meta in rotulos:
    l, a = meta.shape
    pontos = np.zeros((l * a, 3), np.float32)
    for i in range(l * a):
        pontos[i, 0] = i // a
        pontos[i, 1] = i % a
        pontos[i, 2] = 0
    pontos_mundo.append(pontos)

'''
Pontos_mundo é uma matriz nx3 em que n é o número total de cantos encontrados em todas as imagens. 
Cada linha da matriz representa as coordenadas 3D de um canto do tabuleiro de xadrez.
'''

Considerando que a imagem tem 1296 x 972 pixels e que o fabricante da camera NoIR diz que ela tem um campo de visao diagonal de aproximadamente 100 graus, calcule a estimativa inicial para a matriz intrinseca.

In [ ]:
w, h = 1296, 972
theta = math.radians(100)
f = math.sqrt(w**2 + h**2) / (2 * math.tan(theta / 2))

mtx_inicial = np.float64([
    [f, 0, w / 2],
    [0, f, h / 2],
    [0, 0, 1],
])

print(mtx_inicial)

ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    pontos_mundo,
    pontos_imagem,
    (1296, 972),
    mtx_inicial,
    None,
    flags=cv2.CALIB_USE_INTRINSIC_GUESS
    + cv2.CALIB_RATIONAL_MODEL
    + cv2.CALIB_THIN_PRISM_MODEL,
)

print('Erro RMS:', ret)
print('Matriz intrinseca:')
print(mtx)
print('Coeficientes de distorcao:')
print(dist)

Vamos agora testar o erro medio, em pixels, a partir das variaveis rvecs e tvecs, que deve ser inferior a 1.

In [ ]:
erro_total = 0
for i in range(len(pontos_mundo)):
    imgpoints2, _ = cv2.projectPoints(pontos_mundo[i], rvecs[i], tvecs[i], mtx, dist)
    erro = cv2.norm(pontos_imagem[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    erro_total += erro

erro_medio = erro_total / len(pontos_mundo)
print(erro_medio)

Vamos verificar o conteudo da matriz intrinseca da camera.

In [ ]:
fx = mtx[0, 0]
fy = mtx[1, 1]
ox = mtx[0, 2]
oy = mtx[1, 2]
abertura_lateral = math.degrees(2 * math.atan(1296 / (2 * fx)))

print('fx:', fx)
print('fy:', fy)
print('ox:', ox)
print('oy:', oy)
print('centro esperado:', (648, 486))
print('angulo de abertura lateral:', abertura_lateral)

Podemos agora corrigir algumas imagens usando undistort e observar o efeito da funcao.

In [ ]:
imagem = imagens_casadas[0] # Primeira imagem onde o padrão foi encontrado
imagem_corrigida = cv2.undistort(imagem, mtx, dist)

plt.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
plt.show()

plt.imshow(cv2.cvtColor(imagem_corrigida, cv2.COLOR_BGR2RGB))
plt.show()

## 6.7 Determinacao dos parametros extrinsecos: posicao e orientacao da camera

**Atividade fisica fora do notebook.**

Precisamos primeiro ajustar a camera fisicamente:

1. Pivote a junta articulada da camera para baixo ate o seu limite.

2. Verifique que o parafuso de fixacao da camera esta apertado.

3. Posicione o robo em uma superficie plana.

4. Alinhe a borda do papel A4 na qual o padrao de calibracao esta impresso com a borda dianteira do robo.

5. O centro da borda esquerda do papel deve coincidir com o centro da borda dianteira do robo.

6. Ajuste o foco da camera girando o anel externo na montagem da lente. 
Deve ficar focado o centro da imagem (onde estão os quadrados com bola dentro)

7. O padrão de calibração deve estar totalmente visivel na imagem

Vamos encontrar a matriz de transformação homogênea da camera para a IMU.

Capture agora uma imagem com OpenCV. Lembre-se de definir os atributos de resolucao da imagem para 1296 x 972.

In [ ]:
vid = cv2.VideoCapture(0, cv2.CAP_V4L2)
vid.set(cv2.CAP_PROP_AUTOFOCUS, 0)
vid.set(cv2.CAP_PROP_FOCUS, 0)
vid.set(cv2.CAP_PROP_FRAME_WIDTH, 1296)
vid.set(cv2.CAP_PROP_FRAME_HEIGHT, 972)
ret, imagem = vid.read()
vid.release()

if not ret:
    raise RuntimeError('Nao foi possivel capturar imagem com OpenCV.')

cv2.imwrite(str(CAPTURA_EXTRINSECA_PATH), imagem)
plt.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
plt.show()

Converta a imagem para tons de cinza e tente detectar o padrao quadriculado com findChessboardCornersSBWithMeta, como feito na secao 6.6.

Devem ficar visiveis ao menos 13 linhas identificaveis cada uma com 9 pontos!

In [ ]:
gray = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
ret_ext, coordenadas_imagem, atributos = cv2.findChessboardCornersSBWithMeta(
    gray,
    (6, 6),
    flags=cv2.CALIB_CB_EXHAUSTIVE + cv2.CALIB_CB_LARGER + cv2.CALIB_CB_MARKER,
)

print('Padrao encontrado:', ret_ext)
print('Dimensoes dos atributos:', atributos.shape)


Verifique o padrão reconhecido com drawChessboardCorners()

In [ ]:
imagem_vertices = np.array(imagem)
cv2.drawChessboardCorners(imagem_vertices, atributos.shape, coordenadas_imagem, ret_ext)
plt.imshow(cv2.cvtColor(imagem_vertices, cv2.COLOR_BGR2RGB))
plt.show()

Monte um vetor de coordenadas (x', y', z') dos pontos identificados no referencial do robo (lembre-se que z' = 0).

In [ ]:
# O OpenCV devolve os pontos do tabuleiro em uma lista linear.
# Para entender a correspondencia com a grade, vamos transformar essa lista
# em uma tabela com linha, coluna, rotulo e coordenada do pixel na imagem.
linhas, colunas = atributos.shape
pontos_2d = coordenadas_imagem.reshape(-1, 2)

print('Dimensao da grade detectada:')
print('  linhas :', linhas)
print('  colunas:', colunas)
print()
print('Tabela dos pontos detectados:')
print('indice | linha | coluna | rotulo | pixel_x | pixel_y')
for indice in range(linhas * colunas):
    linha = indice // colunas
    coluna = indice % colunas
    rotulo = int(atributos[linha, coluna])
    pixel_x, pixel_y = pontos_2d[indice]
    print(f'{indice:6d} | {linha:5d} | {coluna:6d} | {rotulo:6d} | {pixel_x:7.1f} | {pixel_y:7.1f}')

# Escolha manual da ancora:
# A apostila sugere localizar o ponto no centro da Figura 6.17.
# Esse ponto tem rotulo 2 e coordenadas reais (154,55 mm; 0 mm; 0 mm).
#
# Ajuste os valores abaixo olhando a tabela impressa acima e a imagem com os
# vertices desenhados na celula anterior. Se voce escolher outra ancora, tambem
# ajuste x_ancora_mm e y_ancora_mm para as coordenadas reais desse ponto.
linha_ancora = 0      # <-- ajuste aqui
coluna_ancora = 0     # <-- ajuste aqui
x_ancora_mm = 154.55  # coordenada x' real da ancora, em mm
y_ancora_mm = 0.0     # coordenada y' real da ancora, em mm
z_padrao_mm = 0.0     # todos os pontos do padrao estao no plano z' = 0

# Sinais dos eixos da grade:
# Quando a linha da matriz aumenta em +1, o ponto real pode andar no sentido
# positivo ou negativo de x'. Ajuste sinal_linha_para_x se o resultado ficar
# invertido.
#
# Quando a coluna da matriz aumenta em +1, o ponto real pode andar no sentido
# positivo ou negativo de y'. A apostila lembra que y' cresce da esquerda para
# a direita, enquanto os cantos sao enumerados da direita para a esquerda.
sinal_linha_para_x = 1.0    # use +1.0 ou -1.0
sinal_coluna_para_y = -1.0  # use +1.0 ou -1.0

print('\nAncora escolhida:')
print('  linha, coluna:', linha_ancora, coluna_ancora)
print('  rotulo:', int(atributos[linha_ancora, coluna_ancora]))
indice_ancora = linha_ancora * colunas + coluna_ancora
print('  pixel:', pontos_2d[indice_ancora])
print('  coordenada real [mm]:', (x_ancora_mm, y_ancora_mm, z_padrao_mm))

# Montagem das coordenadas reais dos pontos detectados.
# Para cada canto detectado:
# - delta_linha mede quantos quadrados ele esta acima/abaixo da ancora na tabela;
# - delta_coluna mede quantos quadrados ele esta a esquerda/direita da ancora;
# - cada passo vale SQUARE_SIZE_MM = 12,1 mm;
# - z' e sempre zero porque o padrao esta sobre o plano do papel.
coordenadas_global = []
for indice in range(linhas * colunas):
    linha = indice // colunas
    coluna = indice % colunas

    delta_linha = linha - linha_ancora
    delta_coluna = coluna - coluna_ancora

    x = x_ancora_mm + sinal_linha_para_x * delta_linha * SQUARE_SIZE_MM
    y = y_ancora_mm + sinal_coluna_para_y * delta_coluna * SQUARE_SIZE_MM
    z = z_padrao_mm

    coordenadas_global.append([x, y, z])

coordenadas_global = np.float32(coordenadas_global)

print('\nPrimeiras coordenadas reais montadas:')
for indice in range(min(10, len(coordenadas_global))):
    print(indice, coordenadas_global[indice])

# Com as correspondencias coordenadas_global <-> coordenadas_imagem definidas,
# a chamada solvePnP calcula os parametros extrinsecos.
ret_pnp, rot, trans = cv2.solvePnP(coordenadas_global, coordenadas_imagem, mtx, dist)
matriz_rot, jacobiano = cv2.Rodrigues(rot)

print('\nsolvePnP:', ret_pnp)
print('Matriz de rotacao:')
print(matriz_rot)
print('Vetor de translacao:')
print(trans)

# Conferencias sugeridas pela apostila.
# Se estes avisos aparecerem, volte aos valores manuais acima e confira:
# - se a ancora esta correta;
# - se os sinais dos eixos estao corretos;
# - se o alinhamento fisico do robo com a folha esta bom.
if matriz_rot[0, 1] < 0.95:
    print('Atencao: R[0,1] menor que 0.95. Verifique ancora, sinais e alinhamento.')
if abs(float(trans[0])) > 5:
    print('Atencao: trans[0] maior que 5 mm em modulo. Verifique ancora, sinais e alinhamento.')

## 6.8 Simulando a projecao de pontos e um exemplo de realidade aumentada

## Tarefa - Calcule este produto e determine a posicao das coordenadas na imagem correspondentes ao ponto central do padrao de calibracao (154,55; 0; 0).

In [ ]:
P = np.float64([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
])

G = np.eye(4)
G[:3, :3] = matriz_rot
G[:3, 3] = trans.reshape(3)

MPG = mtx @ P @ G
ponto = np.float64([154.55, 0, 0, 1])
projetado = MPG @ ponto
ponto_x, ponto_y = (projetado[:2] / projetado[2]).astype(int)

print('MPG:')
print(MPG)
print('Coordenadas na imagem:', ponto_x, ponto_y)

plt.imshow(cv2.cvtColor(imagem[ponto_y - 25:ponto_y + 25, ponto_x - 25:ponto_x + 25], cv2.COLOR_BGR2RGB))
plt.show()

## Tarefa - Use as funcoes projectPoints e line para desenhar sobre a imagem de calibracao.

In [ ]:
lado = 2 * SQUARE_SIZE_MM
cx, cy = 154.55, 0

cubo = np.float32([
    [cx - lado / 2, cy - lado / 2, 0],
    [cx + lado / 2, cy - lado / 2, 0],
    [cx + lado / 2, cy + lado / 2, 0],
    [cx - lado / 2, cy + lado / 2, 0],
    [cx - lado / 2, cy - lado / 2, -lado],
    [cx + lado / 2, cy - lado / 2, -lado],
    [cx + lado / 2, cy + lado / 2, -lado],
    [cx - lado / 2, cy + lado / 2, -lado],
])

projetados, jac = cv2.projectPoints(cubo, rot, trans, mtx, dist)
projetados = np.int32(projetados.reshape(-1, 2))

imagem_ar = np.array(imagem)
arestas = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]
for a, b in arestas:
    cv2.line(imagem_ar, tuple(projetados[a]), tuple(projetados[b]), (255, 0, 255), 4)

origem = cubo[0]
eixos = np.float32([origem, origem + [lado, 0, 0], origem + [0, lado, 0], origem + [0, 0, -lado]])
eixos_proj, jac = cv2.projectPoints(eixos, rot, trans, mtx, dist)
eixos_proj = np.int32(eixos_proj.reshape(-1, 2))

cv2.line(imagem_ar, tuple(eixos_proj[0]), tuple(eixos_proj[1]), (0, 0, 255), 4)
cv2.line(imagem_ar, tuple(eixos_proj[0]), tuple(eixos_proj[2]), (0, 255, 0), 4)
cv2.line(imagem_ar, tuple(eixos_proj[0]), tuple(eixos_proj[3]), (255, 0, 0), 4)

plt.imshow(cv2.cvtColor(imagem_ar, cv2.COLOR_BGR2RGB))
plt.show()

## 6.9 Invertendo a Projecao

###Tarefa - Tarefas da secao 6.9

In [ ]:
pontos = np.float32([
    [0, 0],
    [1279, 0],
    [639, 359],
    [0, 719],
    [1279, 719],
    [639, 719],
])

novos_pontos = cv2.undistortPoints(np.float32(pontos), mtx, dist)
print('Coordenadas de projecao ideal:')
print(novos_pontos.reshape(-1, 2))

centro_camera = -matriz_rot.T @ trans
print('Centro de projecao no referencial global:')
print(centro_camera)

direcoes = []
for p in novos_pontos.reshape(-1, 2):
    X = np.float64([[p[0]], [p[1]], [1]])
    direcao = matriz_rot.T @ X
    direcoes.append(direcao)

print('Reta do ponto central:')
print('X(lambda) =')
print(centro_camera)
print('+ lambda *')
print(direcoes[2])

direcao_inferior_central = direcoes[5]
lambda_z0 = -centro_camera[2, 0] / direcao_inferior_central[2, 0]
ponto_z0 = centro_camera + lambda_z0 * direcao_inferior_central
print('Cruzamento do ponto inferior central com z=0:')
print(ponto_z0)

for nome, direcao in zip(['inferior esquerdo', 'inferior central', 'inferior direito'], [direcoes[3], direcoes[5], direcoes[4]]):
    lambda_z0 = -centro_camera[2, 0] / direcao[2, 0]
    ponto_z0 = centro_camera + lambda_z0 * direcao
    print(nome, ponto_z0.reshape(3))

v1 = direcoes[0].reshape(3)
v2 = direcoes[4].reshape(3)
cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
angulo = math.degrees(math.acos(cos_theta))
print('Angulo entre as retas:', angulo)

## 6.9.1 Um caso especial: z' = 0

###Tarefa - A partir da matriz R e do vetor t, calcule a matriz S de projecao reversa para z' = 0

In [ ]:
P = np.float64([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
])

G = np.eye(4)
G[:3, :3] = matriz_rot
G[:3, 3] = trans.reshape(3)

N = P @ G
Nz = N[:, [0, 1, 3]]
Q = np.linalg.inv(Nz)
s = Q / Q[2, 2]

print(s)

###Tarefa - Na secao 6.8 voce calculou as coordenadas da imagem correspondentes ao ponto (154,55, 0), 0, 0). Vamos agora inverter este calculo.

In [ ]:
ponto_imagem = np.float32([[ponto_x, ponto_y]])
ponto_ideal = cv2.undistortPoints(ponto_imagem, mtx, dist).reshape(2)

homogeneo = s @ np.float64([ponto_ideal[0], ponto_ideal[1], 1])
reconstruido = homogeneo[:2] / homogeneo[2]

print('Ponto ideal:', ponto_ideal)
print('Coordenadas reconstruidas:', reconstruido)
print('Valor esperado:', [154.55, 0])

###Tarefa - Reproduza o processo de reconstrucao do Logotipo com uma imagem qualquer a sua escolha. Tente variar os parametros mtx2 e dim e observe que efeito eles produzem na imagem final.

In [ ]:
map_x, map_y = cv2.initUndistortRectifyMap(
    mtx,
    dist,
    s,
    np.float32([[1, 0, 0], [0, 1, 0], [0, 0, 1]]),
    (297, 210),
    cv2.CV_32FC1,
)

nova_imagem = cv2.remap(imagem, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
plt.imshow(cv2.cvtColor(nova_imagem, cv2.COLOR_BGR2RGB))
plt.show()

map_x2, map_y2 = cv2.initUndistortRectifyMap(
    mtx,
    dist,
    s,
    np.float32([[4, 0, 0], [0, 4, -4 * 105], [0, 0, 1]]),
    (4 * 297, 4 * 210),
    cv2.CV_32FC1,
)

nova_imagem2 = cv2.remap(imagem, map_x2, map_y2, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
plt.imshow(cv2.cvtColor(nova_imagem2, cv2.COLOR_BGR2RGB))
plt.show()

Salvar os resultados para uso posterior.

In [ ]:
np.savez(
    RESULTS_PATH,
    image_size=np.array(IMAGE_SIZE),
    mtx=mtx,
    dist=dist,
    rot=rot,
    trans=trans,
    matriz_rot=matriz_rot,
    s=s,
    erro_rms=ret,
    erro_medio_reprojecao=erro_medio,
)

print('Resultados salvos em:', RESULTS_PATH)